https://gyro-interp.readthedocs.io/en/latest/examples.html

In [ ]:
import pandas as pd

# see youth_indicators_tois.ipynb
data = pd.read_csv('../data/youth_indicators_tois.csv')
data

In [ ]:
d=data.query("TOI==200.01").squeeze()
d

In [ ]:
Prot, Prot_err = d.prot, 0.1
Teff, Teff_err = d.teff, 100

## gyro-interp models

In [ ]:
from gyrointerp.models import slow_sequence
from matplotlib.colors import ListedColormap
import numpy as np
import matplotlib.pyplot as plt

tmin, tmax = 100, 1000 #gyro_age.age_med.min(), gyro_age.age_med.max()
Tmin, Tmax = 3800, 6200
slow_seq_ages = [80, 120, 300, 500]
Nages = len(slow_seq_ages)
Teff_grid = np.linspace(Tmin, Tmax, 100)
colors = ListedColormap(plt.cm.viridis(np.linspace(tmin/tmax, 1, Nages))).colors

fig, ax = plt.subplots()

for i,slow_seq_age in enumerate(slow_seq_ages):
    Prot_model = slow_sequence(
        Teff_grid, slow_seq_age, poly_order=7,
        interp_method='pchip_m67'
    )
    ax.plot(
        Teff_grid, Prot_model, color=colors[i], linewidth=3, zorder=-1, label=f"{slow_seq_age}"
    )

# this work
# s = ax.scatter(gyro_age.Teff, gyro_age.Prot, marker='.', c=gyro_age.age_med)
# yerr = gyro_age.Prot_
# s = ax.errorbar(gyro_age.Teff, gyro_age.Prot, yerr=[yerrf], xerr=xerr, marker='.', c=gyro_age.age_med)
# fig.colorbar(s, label='Age [Myr]')
teff = 6000
ax.axvline(teff, 0, 1, ls='--', c='k', label='TOI-6557')
ax.update({
    'xlabel': 'Teff [K]',
    'ylabel': 'Rotation period [days]',
    'title': 'Gyro models from Bouma et al. 2023',
    'xlim': [3500, 6500],
    });
ax.xaxis.set_inverted(True)
ax.legend(title='Age [Myr]', loc='best')

## gyro-interp

In [ ]:
import numpy as np
from gyrointerp import gyro_age_posterior

# uniformly spaced grid between 0 and 2600 megayears
age_grid = np.linspace(0, 2600, 500)

# calculate the age posterior - takes ~30 seconds
age_posterior = gyro_age_posterior(
  Prot, Teff, Prot_err=Prot_err, Teff_err=Teff_err, age_grid=age_grid
)

# calculate dictionary of summary statistics
from gyrointerp import get_summary_statistics
result = get_summary_statistics(age_grid, age_posterior)

print(f"Age = {result['median']} +{result['+1sigma']} -{result['-1sigma']} Myr.")

## age of young cluster planet candidates

In [ ]:
import pandas as pd

loc = '~/github/research/project/young_ttvs/code/tois_yuanzhe'
candidates = pd.read_csv(f'{loc}/tois_in_cluster_candidates_yuanzhe.txt', delimiter='\t')
candidates.shape

In [ ]:
from tqdm import tqdm

age_grid = np.linspace(0, 2600, 500)

for i,row in tqdm(candidates.iterrows()):
    # print(row.TOI)
    toi = row.TOI
    d=data.query("TOI==@toi").squeeze()
    if len(d)>0:
        Prot, Prot_err = d.prot, 0.1
        Teff, Teff_err = d.teff, 100
        if (str(Prot)!='nan') and (str(Teff)!='nan'):
            print(f"Prot={Prot} d, Teff={Teff} K")
            
            # calculate the age posterior - takes ~30 seconds
            age_posterior = gyro_age_posterior(
              Prot, Teff, Prot_err=Prot_err, Teff_err=Teff_err, age_grid=age_grid
            )
            
            # calculate dictionary of summary statistics
            result = get_summary_statistics(age_grid, age_posterior)
            
            print(f"TOI-{toi}: Age = {result['median']} +{result['+1sigma']} -{result['-1sigma']} Myr.")
            break

## custom

In [ ]:
from gyrointerp.plotting import plot_prot_vs_teff

# write the results to the current working directory
outdir = "./"

# show these cluster Prot vs Teff datasets
reference_clusters = [
    'α Per', 'Pleiades', 'Blanco-1', 'Psc-Eri', 'NGC-3532', 'Group-X',
    'Praesepe', 'NGC-6811'
]

# underplot these polynomial fits
model_ids = [
    'α Per', '120-Myr', '300-Myr', 'Praesepe', 'NGC-6811'
]

# overplot these stars with big markers
custom_stardict = {
    "Kepler-1643": {"Prot":5.1, "Teff":4916, "m":"s", "c":"red"},
    "TOI-1136": {"Prot":8.7, "Teff":5770, "m":"X", "c":"pink"},
    "TOI-1937 A": {"Prot":6.6, "Teff":5798, "m":"P", "c":"aqua"},
}

# make the plot
plot_prot_vs_teff(
    outdir, reference_clusters=reference_clusters, model_ids=model_ids,
    custom_stardict=custom_stardict, writepdf=0
)